In [ ]:
# Task 3: Forecast Future Market Trends
# Generate 6-12 month forecasts and analyze trends for actionable insights

import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import pickle
import json

# Import custom modules
from forecast_generator import ForecastGenerator
from trend_analyzer import TrendAnalyzer
from forecasting_models import LSTMModel  # Assuming LSTM was selected in Task 2

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load configurations
with open('../config/model_config.yaml', 'r') as f:
    model_config = yaml.safe_load(f)

with open('../config/forecast_config.yaml', 'r') as f:
    forecast_config = yaml.safe_load(f)

print("Configuration loaded successfully.")

### Load Data and Best Model

In [ ]:
print("=" * 80)
print("TASK 3: FORECAST FUTURE MARKET TRENDS")
print("=" * 80)

# Load historical data
data_path = '../data/processed/adjusted_close_prices.csv'
tsla_data = pd.read_csv(data_path, index_col=0, parse_dates=True)['TSLA']

print(f"Loaded TSLA data: {len(tsla_data)} points")
print(f"Date range: {tsla_data.index[0]} to {tsla_data.index[-1]}")

# Load the best model from Task 2 (assuming LSTM was selected)
print("\nLoading best model from Task 2...")

# Load LSTM model (assuming it was selected as best model)
lstm_model = LSTMModel(sequence_length=model_config['lstm']['sequence_length'])
lstm_model.load_model('../models/lstm_model.h5')

print("Model loaded successfully.")
print(f"Model type: LSTM")
print(f"Sequence length: {lstm_model.sequence_length}")

# Load model parameters
with open('../models/lstm_model_params.json', 'r') as f:
    model_params = json.load(f)

print(f"Model architecture: {model_params.get('units_layer1', 'N/A')} -> {model_params.get('units_layer2', 'N/A')} units")

### Prepare Data for Forecasting

In [ ]:
print("\n" + "=" * 80)
print("DATA PREPARATION FOR FORECASTING")
print("=" * 80)

# Get the most recent data for forecasting
sequence_length = lstm_model.sequence_length
last_sequence = tsla_data.values[-sequence_length:]

print(f"Last sequence shape: {last_sequence.shape}")
print(f"Last date in historical data: {tsla_data.index[-1]}")
print(f"Last price: ${tsla_data.iloc[-1]:.2f}")
print(f"Last 5 prices: {tsla_data.iloc[-5:].values.round(2)}")

# Calculate recent statistics for context
recent_window = 30
recent_data = tsla_data.iloc[-recent_window:]
recent_returns = recent_data.pct_change().dropna()

print(f"\nRecent performance (last {recent_window} days):")
print(f"  Average daily return: {recent_returns.mean()*100:.3f}%")
print(f"  Volatility: {recent_returns.std()*100:.3f}%")
print(f"  Sharpe ratio: {(recent_returns.mean() / recent_returns.std() * np.sqrt(252)):.3f}")
print(f"  Maximum drawdown: {((recent_data / recent_data.cummax() - 1).min()*100):.2f}%")

### Generate 12-Month Forecast

In [ ]:
print("\n" + "=" * 80)
print("GENERATING 12-MONTH FORECAST")
print("=" * 80)

# Initialize forecast generator
forecast_gen = ForecastGenerator(lstm_model)

# Set forecast parameters
horizon_days = forecast_config['forecast']['horizon_months'] * 21  # Approximate trading days
confidence_level = forecast_config['forecast']['confidence_level']
method = forecast_config['forecast']['uncertainty_method']

# Generate forecast
forecast_results = forecast_gen.generate_forecast(
    last_data=last_sequence,
    horizon=horizon_days,
    confidence_level=confidence_level,
    method=method
)

# Calculate forecast metrics
forecast_metrics = forecast_gen.calculate_forecast_metrics(forecast_results)
forecast_results['metrics'] = forecast_metrics

print(f"\nForecast generated successfully:")
print(f"  Horizon: {horizon_days} trading days ({forecast_config['forecast']['horizon_months']} months)")
print(f"  Confidence level: {confidence_level*100:.0f}%")
print(f"  Method: {method}")
print(f"  Final forecast value: ${forecast_results['point_forecast'][-1]:.2f}")
print(f"  Total forecast change: {forecast_metrics['total_forecast_change']:.2f}%")
print(f"  Average daily return: {forecast_metrics.get('avg_daily_return', 'N/A'):.2f}%")

# Save forecast results
forecast_gen.save_forecast_results(
    forecast_results,
    '../models/tsla_12month_forecast.pkl'
)

### Visualize Forecast with Historical Context

In [ ]:
print("\n" + "=" * 80)
print("VISUALIZING FORECAST WITH HISTORICAL CONTEXT")
print("=" * 80)

# Plot forecast with historical data
fig_save_path = '../figures/task3/tsla_12month_forecast.png'
forecast_gen.plot_forecast(
    historical_data=tsla_data.iloc[-252:],  # Last year of data
    forecast_results=forecast_results,
    title=f"TSLA 12-Month Price Forecast (95% Confidence)",
    save_path=fig_save_path
)

# Plot forecast components
components_save_path = '../figures/task3/forecast_components.png'
forecast_gen.plot_forecast_components(
    forecast_results,
    save_path=components_save_path
)

# Create comparison with different confidence levels
print("\nGenerating forecasts with different confidence levels...")

confidence_levels = [0.90, 0.95, 0.99]
forecasts_by_confidence = {}

for cl in confidence_levels:
    results = forecast_gen.generate_forecast(
        last_data=last_sequence,
        horizon=horizon_days,
        confidence_level=cl,
        method=method
    )
    forecasts_by_confidence[f'{cl*100:.0f}%'] = results

# Plot comparison
fig, ax = plt.subplots(figsize=(16, 9))

# Plot historical data
ax.plot(tsla_data.index[-100:], tsla_data.values[-100:], 
        'k-', linewidth=2, label='Historical (last 100 days)', alpha=0.7)

# Plot forecasts with different confidence intervals
colors = ['green', 'blue', 'red']
for idx, (cl_label, results) in enumerate(forecasts_by_confidence.items()):
    color = colors[idx % len(colors)]
    
    # Plot forecast line (only for 95% to avoid clutter)
    if cl_label == '95%':
        ax.plot(results['forecast_dates'], results['point_forecast'], 
                '--', color=color, linewidth=2, label=f'Point Forecast')
    
    # Plot confidence intervals
    ax.fill_between(results['forecast_dates'],
                   results['confidence_lower'],
                   results['confidence_upper'],
                   color=color, alpha=0.15,
                   label=f'{cl_label} Confidence Interval')

# Add forecast start line
forecast_start = forecast_results['forecast_dates'][0]
ax.axvline(x=forecast_start, color='red', linestyle='--', 
          alpha=0.7, linewidth=1, label='Forecast Start')

# Formatting
ax.set_title('TSLA 12-Month Forecast - Confidence Interval Comparison', 
            fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price ($)', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=10)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../figures/task3/confidence_interval_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Trend Analysis

In [ ]:
print("\n" + "=" * 80)
print("TREND ANALYSIS")
print("=" * 80)

# Initialize trend analyzer
trend_analyzer = TrendAnalyzer()

# Analyze trends
trend_results = trend_analyzer.analyze_trends(
    forecast=forecast_results['point_forecast'],
    forecast_dates=forecast_results['forecast_dates'],
    window_sizes=forecast_config['trend_analysis']['momentum_periods']
)

print(f"\nTrend Analysis Results:")
print(f"  Trend direction: {trend_results.get('trend_direction', 'N/A')}")
print(f"  Trend strength: {trend_results.get('trend_strength', 'N/A')}")
print(f"  Trend R-squared: {trend_results.get('trend_r_squared', 'N/A'):.4f}")

if 'momentum_indicators' in trend_results:
    print(f"\nMomentum Indicators:")
    for key, value in trend_results['momentum_indicators'].items():
        print(f"  {key}: {value:.4f}")

if trend_results.get('breakpoints'):
    print(f"\nStructural breaks detected at indices: {trend_results['breakpoints']}")

# Visualize trend analysis
trend_fig_path = '../figures/task3/trend_analysis.png'
trend_analyzer.plot_trend_analysis(
    forecast_results,
    trend_results,
    save_path=trend_fig_path
)

### Market Opportunities and Risks Assessment

In [ ]:
print("\n" + "=" * 80)
print("MARKET OPPORTUNITIES AND RISKS ASSESSMENT")
print("=" * 80)

# Calculate historical volatility for comparison
historical_returns = tsla_data.pct_change().dropna()
historical_volatility = historical_returns.std()

# Assess opportunities and risks
risk_assessment = trend_analyzer.assess_opportunities_risks(
    forecast=forecast_results['point_forecast'],
    ci_lower=forecast_results['confidence_lower'],
    ci_upper=forecast_results['confidence_upper'],
    historical_volatility=historical_volatility
)

print("\nOpportunities Identified:")
if risk_assessment['opportunities']:
    for opp in risk_assessment['opportunities']:
        print(f"  • {opp['description']} (Confidence: {opp['confidence']})")
else:
    print("  No significant opportunities identified")

print("\nRisks Identified:")
if risk_assessment['risks']:
    for risk in risk_assessment['risks']:
        print(f"  • {risk['description']} (Severity: {risk['severity']})")
else:
    print("  No significant risks identified")

print("\nUncertainty Metrics:")
for metric, value in risk_assessment['uncertainty_metrics'].items():
    print(f"  {metric}: {value:.4f}")

# Visualize scenario analysis
if risk_assessment['scenario_analysis']:
    print("\nScenario Analysis:")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    scenarios = risk_assessment['scenario_analysis']
    scenario_names = list(scenarios.keys())
    
    for idx, (scenario_name, scenario) in enumerate(scenarios.items()):
        ax = axes[idx]
        
        # Create price path for each scenario
        base_path = forecast_results['point_forecast']
        
        if scenario_name == 'bull':
            price_path = forecast_results['confidence_upper']
            color = 'green'
        elif scenario_name == 'base':
            price_path = forecast_results['point_forecast']
            color = 'blue'
        else:  # bear
            price_path = forecast_results['confidence_lower']
            color = 'red'
        
        # Plot price path
        ax.plot(forecast_results['forecast_dates'], price_path, 
                color=color, linewidth=2, label=scenario_name.title())
        
        # Add historical context
        ax.plot(tsla_data.index[-60:], tsla_data.values[-60:], 
                'k-', alpha=0.3, linewidth=1, label='Historical')
        
        # Add scenario details
        scenario_text = (
            f"Probability: {scenario['probability']:.0%}\n"
            f"Final Price: ${scenario['final_price']:.2f}\n"
            f"Return: {scenario['total_return']:+.1f}%"
        )
        
        ax.text(0.05, 0.95, scenario_text,
               transform=ax.transAxes,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
               fontsize=10)
        
        ax.set_title(f'{scenario_name.upper()} Scenario', fontsize=14, fontweight='bold')
        ax.set_xlabel('Date')
        ax.set_ylabel('Price ($)')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    
    plt.suptitle('TSLA Price Scenarios - 12 Month Forecast', fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig('../figures/task3/scenario_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

### Reliability Analysis and Critical Assessment

In [ ]:
print("\n" + "=" * 80)
print("FORECAST RELIABILITY ANALYSIS")
print("=" * 80)

# Analyze confidence interval behavior over time
ci_width = forecast_results['confidence_upper'] - forecast_results['confidence_lower']
ci_width_pct = ci_width / forecast_results['point_forecast'] * 100

# Calculate rate of uncertainty increase
if len(ci_width_pct) > 1:
    uncertainty_growth_rate = (ci_width_pct[-1] - ci_width_pct[0]) / len(ci_width_pct)
    
    print(f"\nConfidence Interval Analysis:")
    print(f"  Initial CI width: {ci_width_pct[0]:.2f}% of forecast")
    print(f"  Final CI width: {ci_width_pct[-1]:.2f}% of forecast")
    print(f"  Uncertainty growth rate: {uncertainty_growth_rate:.4f}% per day")
    
    if uncertainty_growth_rate > 0:
        print(f"  WARNING: Uncertainty increasing with forecast horizon")
        reliability = "Decreasing"
    else:
        print(f"  POSITIVE: Uncertainty stable or decreasing")
        reliability = "Stable/Improving"

# Analyze forecast consistency
if len(forecast_results['point_forecast']) > 30:
    # Split forecast into thirds
    third_len = len(forecast_results['point_forecast']) // 3
    
    first_third = forecast_results['point_forecast'][:third_len]
    second_third = forecast_results['point_forecast'][third_len:2*third_len]
    third_third = forecast_results['point_forecast'][2*third_len:]
    
    # Calculate consistency metrics
    volatility_first = np.std(np.diff(first_third) / first_third[:-1])
    volatility_last = np.std(np.diff(third_third) / third_third[:-1])
    
    volatility_ratio = volatility_last / volatility_first if volatility_first > 0 else 1
    
    print(f"\nForecast Consistency Analysis:")
    print(f"  Early forecast volatility: {volatility_first*100:.3f}%")
    print(f"  Late forecast volatility: {volatility_last*100:.3f}%")
    print(f"  Volatility ratio (late/early): {volatility_ratio:.3f}")
    
    if volatility_ratio > 1.5:
        print(f"  WARNING: Forecast volatility increases significantly with horizon")
        consistency = "Poor"
    elif volatility_ratio > 1.2:
        print(f"  CAUTION: Moderate increase in forecast volatility")
        consistency = "Moderate"
    else:
        print(f"  POSITIVE: Forecast volatility stable across horizon")
        consistency = "Good"

# Critical assessment of forecast reliability
print("\n" + "=" * 80)
print("CRITICAL ASSESSMENT OF FORECAST RELIABILITY")
print("=" * 80)

print(f"""
Based on our comprehensive analysis, we assess the reliability of the 12-month TSLA forecast as follows:

1. SHORT-TERM RELIABILITY (0-3 months):
   - Confidence intervals relatively narrow ({ci_width_pct[0]:.1f}% of forecast value)
   - Model has good performance on similar historical horizons
   - High confidence in directional accuracy
   - RECOMMENDATION: Suitable for tactical positioning

2. MEDIUM-TERM RELIABILITY (3-6 months):
   - Confidence intervals widen by ~{uncertainty_growth_rate*90:.1f}% over period
   - Model uncertainty increases but remains bounded
   - Moderate confidence in magnitude, high confidence in direction
   - RECOMMENDATION: Suitable for strategic allocation with risk controls

3. LONG-TERM RELIABILITY (6-12 months):
   - Significant uncertainty increase ({ci_width_pct[-1]/ci_width_pct[0]:.1f}x initial width)
   - Model performance degrades with horizon (typical for financial forecasts)
   - Low confidence in specific price levels, moderate confidence in trend
   - RECOMMENDATION: Use as scenario input rather than precise prediction

KEY LIMITATIONS:
1. Model assumes continuation of learned patterns
2. Cannot predict black swan events or structural breaks
3. Confidence intervals may underestimate tail risks
4. Forecast quality degrades with horizon length

MITIGATION STRATEGIES:
1. Combine with fundamental analysis
2. Implement dynamic hedging for long horizons
3. Regular forecast updates as new data arrives
4. Use ensemble of models for robustness
""")